<a href="https://colab.research.google.com/github/Sanchita3005/Cyber-security-/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import re
import math
import logging
from getpass import getpass

logging.basicConfig(
    filename="password_analyzer.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

COMMON_PASSWORDS = {
    "password", "password123", "123456", "12345678",
    "123456789", "qwerty", "qwerty123", "admin",
    "admin123", "letmein", "welcome", "abc123"
}


def validate_input(password):
    if not password:
        return False, "Password cannot be empty."

    if len(password) > 128:
        return False, "Password must not exceed 128 characters."

    if any(ord(ch) < 32 for ch in password):
        return False, "Invalid characters detected."

    return True, ""


def check_password(password):
    score = 0
    suggestions = []

    length = len(password)

    if length >= 8:
        score += 1
    else:
        suggestions.append("Use at least 8 characters.")

    if length >= 12:
        score += 1

    if re.search(r"[a-z]", password):
        score += 1
    else:
        suggestions.append("Add lowercase letters.")

    if re.search(r"[A-Z]", password):
        score += 1
    else:
        suggestions.append("Add uppercase letters.")

    if re.search(r"\d", password):
        score += 1
    else:
        suggestions.append("Add numbers.")

    if re.search(r"[^A-Za-z0-9]", password):
        score += 1
    else:
        suggestions.append("Add special characters.")

    if password.lower() in COMMON_PASSWORDS:
        score = 0
        suggestions.append("Avoid common passwords.")

    if re.search(r"(.)\1{2,}", password):
        score -= 1
        suggestions.append("Avoid repeated characters.")

    if re.search(r"(1234|abcd|qwer)", password.lower()):
        score -= 1
        suggestions.append("Avoid predictable sequences.")

    score = max(0, min(score, 6))

    if score <= 2:
        strength = "WEAK"
    elif score <= 4:
        strength = "MEDIUM"
    else:
        strength = "STRONG"

    charset = 0

    if re.search(r"[a-z]", password):
        charset += 26
    if re.search(r"[A-Z]", password):
        charset += 26
    if re.search(r"\d", password):
        charset += 10
    if re.search(r"[^A-Za-z0-9]", password):
        charset += 32

    entropy = 0

    if charset:
        entropy = round(length * math.log2(charset), 2)

    return strength, score, entropy, suggestions


def main():
    print("=" * 45)
    print("       PASSWORD STRENGTH ANALYZER")
    print("=" * 45)

    password = getpass("Enter password: ")

    valid, message = validate_input(password)

    if not valid:
        print("\nError:", message)
        logging.warning("Invalid password input.")
        return

    strength, score, entropy, suggestions = check_password(password)

    logging.info(
        "Password analyzed: length=%d, strength=%s, score=%d",
        len(password), strength, score
    )

    print("\n--- SECURITY ANALYSIS ---")
    print("Length       :", len(password))
    print("Strength     :", strength)
    print("Score        :", f"{score}/6")
    print("Entropy      :", f"{entropy} bits")

    if strength == "STRONG":
        print("\nPassword is strong.")
    else:
        print("\nSuggestions:")
        for suggestion in suggestions:
            print("-", suggestion)

    print("\nSecurity Checks:")
    print("Lowercase    :", "PASS" if re.search(r"[a-z]", password) else "FAIL")
    print("Uppercase    :", "PASS" if re.search(r"[A-Z]", password) else "FAIL")
    print("Number       :", "PASS" if re.search(r"\d", password) else "FAIL")
    print("Special Char :", "PASS" if re.search(r"[^A-Za-z0-9]", password) else "FAIL")
    print("Minimum 8    :", "PASS" if len(password) >= 8 else "FAIL")

    print("\nNote: Password contents are never written to the log file.")


if __name__ == "__main__":
    main()

       PASSWORD STRENGTH ANALYZER
